# Phase 2 — Local Bina SFT Data Generation

Generates new Bina conversation data using your own already-fine-tuned
`bina-qwen25-7b-dpo-v11` model, running locally via `llama_cpp_python` on this machine's
GPU — zero API cost, per the plan's decision to reject the ~$10,500-$22,500 it would have
cost to generate this via a paid API. **This notebook runs locally, not in Colab** — it needs
the GGUF model files in `C:\Models\` and `character.yaml` from the sibling Bina project,
neither of which exist on a Colab VM. Open and run this directly in VS Code (or `jupyter
notebook`), not uploaded anywhere.

Covers both `stage_2_sft` synthetic buckets from `configs/data_mix.yaml`: the general
"teacher-generated" conversation category, and all five "difficult examples" sub-categories
(topic changes, vague messages, emotional responses, short replies, repetition-avoidance).
Both are resumable — `generate_sft_data.py` counts existing lines in the output file and
picks up from there, so stopping and re-running this notebook (across many sessions, over
however many days/weeks it takes) never duplicates or loses work.

**Needs `llama_cpp_python` with CUDA support.** The sibling project's venv already has a
working build (`C:\Users\alanl\OneDrive\Documents\Bina\bina_chat\.venv\`) —
point this notebook's Jupyter kernel at that venv's `python.exe`, or install
`llama_cpp_python` with CUDA into your own environment first.

## 1. Setup

In [ ]:
import os
import sys

REPO_ROOT = r"C:\Users\alanl\OneDrive\Documents\Bina_LLM_Training"
sys.path.insert(0, os.path.join(REPO_ROOT, "pretrain"))

from bina_pretrain import data_pipeline as dp
from bina_pretrain.generate_sft_data import run, count_existing

print("bina_pretrain package loaded OK")


## 2. Config

Target token counts come from `configs/data_mix.yaml`, same as Phase 2's data-ingestion
notebook — not duplicated here as separate numbers. `generate_sft_data.py`'s `run()` counts
*conversations*, not tokens directly, so `AVG_TOKENS_PER_CONVERSATION` is a rough estimate
(measured from real generated output during development: ~800 tokens/conversation at
`n_turns=3`, system prompt included) used to size how many conversations to aim for — check
the actual yield in §5 after a run and adjust `n_conversations` in a future session if the
estimate was off. No need for precision here: this is a multi-day background process you can
always extend.

In [ ]:
MIX_YAML_PATH = os.path.join(REPO_ROOT, "pretrain", "configs", "data_mix.yaml")
mix_config = dp.load_mix_config(MIX_YAML_PATH)
sft_stage = mix_config["stage_2_sft"]

CHARACTER_YAML = r"C:\Users\alanl\OneDrive\Documents\Bina\bina_chat\config\character.yaml"
BASE_MODEL = r"C:\Models\Qwen2.5-7B-Instruct-Q4_K_M\qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"
LORA_PATH = r"C:\Models\bina-qwen25-7b-dpo-v11-lora\bina-qwen25-7b-dpo-v11-lora-f16.gguf"
OUTPUT_DIR = os.path.join(REPO_ROOT, "pretrain", "_local_drive_stub", "sft_data")
os.makedirs(OUTPUT_DIR, exist_ok=True)

AVG_TOKENS_PER_CONVERSATION = 800  # rough estimate, see markdown above
N_TURNS = 3

teacher_target_tokens = sft_stage["locally_generated_teacher_conversation"]["target_tokens"]
difficult_target_tokens = sft_stage["locally_generated_difficult_examples"]["target_tokens"]
teacher_n_conversations = teacher_target_tokens // AVG_TOKENS_PER_CONVERSATION
difficult_n_conversations = difficult_target_tokens // AVG_TOKENS_PER_CONVERSATION

teacher_out_path = os.path.join(OUTPUT_DIR, "teacher_generated.jsonl")
difficult_out_path = os.path.join(OUTPUT_DIR, "difficult_examples.jsonl")

print(f"teacher_generated: target {teacher_target_tokens:,} tokens ~= {teacher_n_conversations:,} conversations")
print(f"difficult_examples: target {difficult_target_tokens:,} tokens ~= {difficult_n_conversations:,} conversations")
print(f"already have: {count_existing(teacher_out_path)} teacher, {count_existing(difficult_out_path)} difficult")


## 3. Generate: teacher-generated conversation

Runs until `teacher_n_conversations` is reached, resuming from whatever's already in
`teacher_out_path`. This is the long-running cell — expect this to take a real amount of
wall-clock time (development testing measured ~30 tok/s real decode throughput on a 4060, so
budget accordingly and feel free to interrupt the kernel and re-run this cell later; nothing
is lost).

In [ ]:
run(
    category="teacher_generated",
    out_path=teacher_out_path,
    n_conversations=teacher_n_conversations,
    n_turns=N_TURNS,
    seed=0,
    character_yaml_path=CHARACTER_YAML,
    base_model_path=BASE_MODEL,
    lora_path=LORA_PATH,
)


## 4. Generate: difficult examples

Same as above, covering all five sub-categories (topic_change, vague_message,
emotional_response, short_reply, repetition_avoidance), chosen randomly per conversation.

In [ ]:
run(
    category="difficult_examples",
    out_path=difficult_out_path,
    n_conversations=difficult_n_conversations,
    n_turns=N_TURNS,
    seed=1,
    character_yaml_path=CHARACTER_YAML,
    base_model_path=BASE_MODEL,
    lora_path=LORA_PATH,
)


## 5. Check actual token yield

Tokenizes what's actually been generated so far (real count, not the §2 estimate) so you can
judge whether `AVG_TOKENS_PER_CONVERSATION` needs adjusting and whether either category needs
more conversations queued up in a future session.

In [ ]:
import json

from bina_pretrain.sft import build_example

def real_token_count(path):
    if not os.path.exists(path):
        return 0, 0
    total = 0
    n = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            ids, _ = build_example(ex["messages"])
            total += len(ids)
            n += 1
    return total, n

for label, path, target in [
    ("teacher_generated", teacher_out_path, teacher_target_tokens),
    ("difficult_examples", difficult_out_path, difficult_target_tokens),
]:
    tokens, n = real_token_count(path)
    pct = 100 * tokens / target if target else 0
    print(f"{label}: {n:,} conversations, {tokens:,} real tokens ({pct:.2f}% of {target:,} target)")
